In [51]:
import torch 
import torch.nn as nn 
import numpy as np
import math
from torch.nn import functional as F

In [52]:
text = open("./../input.txt","r").read()
chars =sorted(list(set(text)))
vocab_size =len(chars)
print(''.join(chars))
print(vocab_size)
# print(text)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [53]:
stoi = {s:i for i,s in enumerate(chars)}
itos = {i:s for s,i in stoi.items()}
encode = lambda s: [stoi[c] for c in s] 
decode = lambda l: ''.join([itos[c] for c in l])

print(encode("hi there"))
print(decode(encode("hi there")))


[46, 47, 1, 58, 46, 43, 56, 43]
hi there


In [54]:
data = torch.tensor(encode(text) , dtype = torch.long)
data.shape , data.dtype
# data[:]

(torch.Size([1115394]), torch.int64)

In [55]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]
train_data.shape

torch.Size([1003854])

In [56]:
data.shape


torch.Size([1115394])

In [58]:
batch_size = 4
context_len = 20
n_embd = 10
hidden_size = 10 
def get_batches(data ,batch_size , context_len):
    
    # print(len(data))
    ix = torch.randint(len(data)-context_len ,(batch_size,))
    xb = torch.stack([data[i : context_len+i] for i in ix ])
    yb = torch.stack([data[i+1 : context_len+i+1] for i in ix ])
    
    return xb,yb


            
            

embedding = nn.Embedding(vocab_size,n_embd)
xb,yb = get_batches(train_data,batch_size,context_len)
xb.shape , yb.shape        
    
    

(torch.Size([4, 20]), torch.Size([4, 20]))

In [ ]:
class SimpleRnn(nn.Module):
    def __init__(self,input_size, hidden_size ,output_size):
        super(SimpleRnn,self).__init__()
        self.hidden_size = hidden_size
        self.input_size = input_size
        
        # weights
        self.w_ih = nn.Linear(input_size,hidden_size) 
        self.w_hh = nn.Linear(hidden_size,hidden_size) 
        self.w_hy = nn.Linear(hidden_size,output_size) 
    
    def forward(self,x,hprev):
        ht = torch.tanh(self.w_ih(x) +self.w_hh(hprev) )
        
        yt = self.w_hy(ht)
        # print(yt.shape)
        
        return yt,ht
    
    def init_hidden(self,batch_size):
        return torch.zeros(batch_size,self.hidden_size)

In [63]:
m = SimpleRnn(n_embd,hidden_size ,vocab_size)
optimizer = torch.optim.AdamW(m.parameters(),lr =1e-3)

for i in range (10000):
    xb,yb = get_batches(train_data,batch_size,context_len)
    
    h = m.init_hidden(batch_size)
    loss =0.0
    
    for t in range(context_len):
        x_embd = embedding(xb[:,t])
        # print(x_embd.shape)
        logit , h = m(x_embd,h)
        # print(logit)
        loss += F.cross_entropy(logit, yb[:,t])


    loss/=context_len    
    optimizer.zero_grad(set_to_none = True)
    
    loss.backward()
    torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
    
    optimizer.step()
    if i%1000==0:
        print(f"{i} :{loss}")
    
    


0 :4.118210792541504
1000 :2.929384708404541
2000 :2.7975244522094727
3000 :2.588404893875122
4000 :2.660926580429077
5000 :2.67867112159729
6000 :2.7447702884674072
7000 :2.5664961338043213
8000 :2.3618550300598145
9000 :2.719585657119751


tensor([[ 7.5332e-03,  3.4704e+00, -1.3541e+00, -1.1489e+01, -9.8031e+00,
         -1.7745e-01,  9.6829e-01, -1.2159e+00,  1.3525e-01, -8.3582e+00,
         -6.1167e-01, -9.6385e-01, -1.2479e+00, -2.2586e+00, -3.8063e+00,
         -3.1149e+00, -4.3057e+00, -2.4339e+00, -4.7900e+00, -3.3040e+00,
         -4.2092e+00, -2.1823e+00, -6.7013e+00, -4.8274e+00, -2.8832e+00,
         -4.2165e+00, -1.9675e+00, -2.5119e+00, -5.0714e+00, -7.8290e+00,
         -2.5428e+00, -3.0237e+00, -2.8439e+00, -3.0316e+00, -5.1887e+00,
         -5.2988e+00, -6.3463e+00, -4.3408e+00, -5.6780e+00,  1.7137e+00,
         -7.1054e-01,  1.0090e+00,  1.0091e+00,  2.6040e+00, -6.9509e-02,
          4.8731e-01,  8.1359e-01,  1.1957e+00, -3.9032e+00, -5.0148e-01,
          1.4202e+00,  9.5637e-01,  2.4435e+00,  6.9387e-01,  1.1409e-01,
         -3.4467e+00,  3.0301e+00,  2.3948e+00,  2.2181e+00,  2.0627e-01,
          2.8878e-02, -1.5955e-01, -2.6803e+00,  5.1232e-01, -3.6390e+00],
        [-1.4566e+00,  1.1646e+00, -2

In [76]:
idx = torch.zeros((1,1), dtype=torch.long )
# x_t = embedding(idx)
h =m.init_hidden(1)
print(h.shape)
output = [idx.item()]
for i in range(200):
    x_t = embedding(idx[:,0])
    # print(x_t.shape)
    logits , h =m(x_t,h)
    probs = F.softmax(logits ,dim=-1)
    idx_next = torch.multinomial(probs, num_samples=1)
    output += [idx_next.item()]
    idx =idx_next
    
    
print(decode(output))
    
    
    
    
    
    

torch.Size([1, 10])

Iow
Dek heran.

My usei
Narlb. Ilas,
Wn Oo chutw?ene ser in os; b ayak,
Rs osdesuu, fofe ire wou tinre havanswinists nhare beyeg dowsceg dor tine tinr lers,
Woden.
VIhe than eg, ynfarires anady; forar
